# GDAL/OGR Command-Line Geospatial Workflow

This notebook documents a complete raster and vector processing pipeline
using GDAL/OGR command-line tools — the industry-standard library underlying
most geospatial software including QGIS, ArcGIS, and Python's rasterio.

## Why CLI GDAL matters alongside Python
- Faster for single operations (no Python overhead)
- Scriptable in bash for server/HPC environments without Jupyter
- Direct access to GDAL drivers and creation options (COG, compression)
- Essential for `gdal_calc.py`, format conversion, and batch reprojection

In [ ]:
import subprocess
import rasterio
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np

def run_gdal(cmd, show_output=True):
    """Run a GDAL command and display output cleanly."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if show_output:
        print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
    return result

%matplotlib inline

## Step 1: Inspect Raw Raster


In [ ]:
run_gdal("gdalinfo -stats data/raw/ghi_col.tif")

## Step 2: Reproject Raster to WGS84 UTM Zone 13N

The Solargis GHI raster is originally in NAD83 / UTM Zone 13N (EPSG:26913).

In this step, we standardize the dataset to WGS84 / UTM Zone 13N (EPSG:32613) for interoperability with other geospatial datasets and web mapping workflows.

In [ ]:
run_gdal("bash scripts/02_reproject.sh data/raw/solargis_ghi_colorado.tif")

import rasterio
import matplotlib.pyplot as plt
import numpy as np

# Read rasters
input_path = "data/raw/solargis_ghi_colorado.tif"
output_path = "data/processed/raster_utm13n.tif"

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, path, title in [
    (axes[0], input_path, "Original: NAD83 / UTM 13N (EPSG:26913)"),
    (axes[1], output_path, "Reprojected: WGS84 / UTM 13N (EPSG:32613)"),
]:

    with rasterio.open(path) as src:
        data = src.read(1).astype(float)

        # handle nodata safely
        if src.nodata is not None:
            data[data == src.nodata] = np.nan

        ax.imshow(data, cmap="YlOrRd", origin="upper")
        ax.set_title(title)
        ax.axis("off")

plt.suptitle("Solargis GHI Reprojection Workflow", fontsize=13)
plt.tight_layout()
plt.show()

## Step 3: Clip to Colorado Boundary

In [ ]:
run_gdal("bash scripts/03_clip.sh data/raw/colorado_boundary.shp")

with rasterio.open("data/processed/raster_clipped.tif") as src:
    data = src.read(1).astype(float)
    data[data == src.nodata] = np.nan
    extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]

boundary = gpd.read_file("data/raw/colorado_boundary.shp").to_crs("EPSG:32613")

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(data, cmap="YlOrRd", extent=extent, origin="upper")
boundary.boundary.plot(ax=ax, color="black", linewidth=0.8)
ax.set_title("Clipped to Colorado State Boundary")
plt.tight_layout()
plt.show()

## Step 4: Normalize Values and Create COG

In [ ]:
run_gdal("bash scripts/04_normalize.sh")
run_gdal("bash scripts/05_cog.sh")

# Show normalized histogram
with rasterio.open("data/processed/raster_normalized.tif") as src:
    data = src.read(1).astype(float)
    data[data == -9999] = np.nan

valid = data[~np.isnan(data)].flatten()
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(valid, bins=50, color="#E8593C", edgecolor="white", alpha=0.85)
ax.set_xlabel("Normalized Value (0–1)")
ax.set_ylabel("Pixel Count")
ax.set_title("Value Distribution After Normalization (gdal_calc)")
plt.tight_layout()
plt.show()

print(f"\nNormalized range: {valid.min():.4f} – {valid.max():.4f}")
print(f"Mean: {valid.mean():.4f}")